# 03 - Cleaning — Calls

**Goal:** clean the Calls log — fix types, drop empty columns, document missing-value logic, and add flags (`is_attended`, `is_zero_duration`) for analysis.

## 1. Load & inspect

In [21]:
import pandas as pd
from pathlib import Path

In [2]:

DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

calls = pd.read_excel(
    DATA_RAW / 'Calls (Done).xlsx',
    dtype={'Id': str, 'CONTACTID': str}
)

In [3]:
calls.shape

(95874, 11)

In [4]:
calls.describe()

,Call Duration (in seconds),Dialled Number,Scheduled in CRM,Tag
count,95791.000000,0.0,86875.000000,0.0
mean,164.977263,NaN,0.001635,NaN
std,401.410826,NaN,0.040397,NaN
min,0.000000,NaN,0.000000,NaN
25%,4.000000,NaN,0.000000,NaN
50%,8.000000,NaN,0.000000,NaN
75%,98.000000,NaN,0.000000,NaN
max,7625.000000,NaN,1.000000,NaN


In [5]:
calls.head()

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN


In [6]:
calls.dtypes

Id                             object
Call Start Time                object
Call Owner Name                object
CONTACTID                      object
Call Type                      object
Call Duration (in seconds)    float64
Call Status                    object
Dialled Number                float64
Outgoing Call Status           object
Scheduled in CRM              float64
Tag                           float64
dtype: object

In [7]:
calls.columns.tolist()

['Id',
 'Call Start Time',
 'Call Owner Name',
 'CONTACTID',
 'Call Type',
 'Call Duration (in seconds)',
 'Call Status',
 'Dialled Number',
 'Outgoing Call Status',
 'Scheduled in CRM',
 'Tag']

In [8]:
calls.isna().sum()

Id                                0
Call Start Time                   0
Call Owner Name                   0
CONTACTID                      3933
Call Type                         0
Call Duration (in seconds)       83
Call Status                       0
Dialled Number                95874
Outgoing Call Status           8999
Scheduled in CRM               8999
Tag                           95874
dtype: int64

In [9]:
calls.nunique() 

Id                            95874
Call Start Time               68445
Call Owner Name                  33
CONTACTID                     15214
Call Type                         3
Call Duration (in seconds)     2619
Call Status                      11
Dialled Number                    0
Outgoing Call Status              4
Scheduled in CRM                  2
Tag                               0
dtype: int64

In [10]:
categorical_cols = ['Id',
 'Call Start Time',
 'Call Owner Name',
 'CONTACTID',
 'Call Type',
 'Call Duration (in seconds)',
 'Call Status',
 'Dialled Number',
 'Outgoing Call Status',
 'Scheduled in CRM',
 'Tag'
]

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(calls[col].value_counts(dropna=False))


=== Id ===
Id
5805028000000805001    1
5805028000000768006    1
5805028000000764027    1
5805028000000787003    1
5805028000000768019    1
                      ..
5805028000056889515    1
5805028000056875317    1
5805028000056832495    1
5805028000056893619    1
5805028000056893631    1
Name: count, Length: 95874, dtype: int64

=== Call Start Time ===
Call Start Time
06.06.2024 15:07    9
09.04.2024 14:23    8
18.03.2024 14:29    7
02.04.2024 18:15    7
12.04.2024 14:51    7
                   ..
21.06.2024 14:57    1
21.06.2024 14:58    1
21.06.2024 14:59    1
04.11.2023 10:45    1
30.06.2023 12:09    1
Name: count, Length: 68445, dtype: int64

=== Call Owner Name ===
Call Owner Name
Yara Edwards       9059
Julia Nelson       7446
Ian Miller         7215
Charlie Davis      7213
Diana Evans        6857
Ulysses Adams      6085
Amy Green          5982
Nina Scott         5581
Victor Barnes      5439
Kevin Parker       5406
Paula Underwood    4580
Quincy Vincent     4384
Jane Smith      

## Initial inspection — Calls

**Size.** 95,874 calls × 11 columns — 4.4× larger than Deals, a detailed log of manager calls.

**Type conversion.** `Id` and `CONTACTID` set to string at load. `Call Start Time` parsed to datetime (`DD.MM.YYYY HH:MM`). `Call Duration (in seconds)` kept as float (NaN support). `Scheduled in CRM` holds 0/1 — left as is (converting to bool would turn NaN into True).

**Columns to drop.** `Tag` and `Dialled Number` are 100% empty (95,874 NaN each) — export artifacts.

**Missing-value decisions.** `CONTACTID` (3,933, ~4%) — kept as NaN, calls not linked to a contact. `Call Duration` (83) — kept as NaN. `Outgoing Call Status` and `Scheduled in CRM` (8,999 each) — kept as NaN: filled only for outbound calls; empty by definition for inbound/missed. The logic checks out: Inbound (3,078) + Missed (5,921) = 8,999, exactly matching the missing count. This is business logic, not a bug.

**Observations for the report.** 22% of calls have Duration = 0 (21,975) — dialed-and-dropped or no connection; an indicator of contact-data quality. 33 unique Call Owners vs 27 in Deals — some managers make calls but don't own deals (likely junior / call-center). Call types: Outbound 91%, Missed 6%, Inbound 3% — the school mostly calls out rather than receiving inbound.

In [11]:
calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], format='%d.%m.%Y %H:%M')
calls = calls.drop(columns=['Tag', 'Dialled Number'])

In [12]:
print(calls['Scheduled in CRM'].value_counts(dropna=False))

Scheduled in CRM
0.0    86733
NaN     8999
1.0      142
Name: count, dtype: int64


In [13]:
calls.dtypes

Id                                    object
Call Start Time               datetime64[ns]
Call Owner Name                       object
CONTACTID                             object
Call Type                             object
Call Duration (in seconds)           float64
Call Status                           object
Outgoing Call Status                  object
Scheduled in CRM                     float64
dtype: object

In [14]:
calls.shape

(95874, 9)

### Date range in Call Start Time

In [15]:
print(f"Call Start Time range:")
print(f"  Min: {calls['Call Start Time'].min()}")
print(f"  Max: {calls['Call Start Time'].max()}")

Call Start Time range:
  Min: 2023-06-30 08:43:00
  Max: 2024-06-21 15:31:00


### Flag: answered vs not answered

In [16]:
# Status mapping: answered (True) / not answered (False)
attended_mapping = {
    # Answered
    'Attended Dialled': True,
    'Received': True,
    'Scheduled Attended': True,
    'Scheduled Attended Delay': True,
    # Not answered
    'Unattended Dialled': False,
    'Missed': False,
    'Cancelled': False,
    'Overdue': False,
    'Scheduled Unattended': False,
    'Scheduled Unattended Delay': False,
    'Scheduled': False,
}

calls['is_attended'] = calls['Call Status'].map(attended_mapping)

# Check
print(calls['is_attended'].value_counts(dropna=False))
print(f"\nShare of answered calls: {calls['is_attended'].sum() / len(calls) * 100:.1f}%")

is_attended
True     73816
False    22058
Name: count, dtype: int64

Share of answered calls: 77.0%


In [17]:
calls['is_zero_duration'] = calls['Call Duration (in seconds)'] == 0
print(f"Calls with Duration = 0: {calls['is_zero_duration'].sum()}")
print(f"Share of all calls: {calls['is_zero_duration'].sum() / len(calls) * 100:.1f}%")

Calls with Duration = 0: 21975
Share of all calls: 22.9%


### Call duration — what's inside

In [18]:
print("Call Duration stats (seconds):")
print(calls['Call Duration (in seconds)'].describe())
print(f"\nMax in minutes: {calls['Call Duration (in seconds)'].max() / 60:.1f}")
print(f"Max in hours: {calls['Call Duration (in seconds)'].max() / 3600:.1f}")

# Calls longer than 1 hour
long_calls = calls[calls['Call Duration (in seconds)'] > 3600]
print(f"\nCalls longer than 1 hour: {len(long_calls)}")
print(f"Calls longer than 2 hours: {(calls['Call Duration (in seconds)'] > 7200).sum()}")
print(f"Calls longer than 3 hours: {(calls['Call Duration (in seconds)'] > 10800).sum()}")

Call Duration stats (seconds):
count    95791.000000
mean       164.977263
std        401.410826
min          0.000000
25%          4.000000
50%          8.000000
75%         98.000000
max       7625.000000
Name: Call Duration (in seconds), dtype: float64

Max in minutes: 127.1
Max in hours: 2.1

Calls longer than 1 hour: 42
Calls longer than 2 hours: 1
Calls longer than 3 hours: 0


## Duplicate check

In [19]:
print(f"Duplicate Ids: {calls['Id'].duplicated().sum()}")
print(f"Fully duplicated rows: {calls.duplicated().sum()}")

Duplicate Ids: 0
Fully duplicated rows: 0


## Save

In [ ]:
calls.to_parquet(DATA_PROCESSED / 'calls_clean.parquet')
print("Parquet saved")

calls.to_excel(DATA_PROCESSED / 'calls_clean.xlsx', index=False)
print("Excel saved")